# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前後を比べる

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
NVIDIA L4, 23034 MiB



In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


In [ ]:

from pathlib import Path
import os, json, time, re
CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,   # 116 件 x 10 周。1 周 15 ステップで合計 150 ステップ
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {"do_sample": False},
    "data": "glossary_sft.jsonl（scripts/make_sft_dataset.py の出力そのまま、58 語 x 2 テンプレート）",
}
WORK = Path("/content/glossary_lab"); WORK.mkdir(parents=True, exist_ok=True)
rows = [json.loads(l) for l in Path("/content/glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(rows), rows[0])


116 {'messages': [{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]}


In [ ]:

EVAL = json.loads(r"""[
 {
  "group": "seen",
  "question": "「LoRA」とは何ですか？",
  "id": "lora",
  "must_include": [
   "低ランク",
   "固定"
  ],
  "query": "LoRA"
 },
 {
  "group": "seen",
  "question": "「QLoRA」とは何ですか？",
  "id": "qlora",
  "must_include": [
   "量子化",
   "LoRA"
  ],
  "query": "QLoRA"
 },
 {
  "group": "seen",
  "question": "「エポック」とは何ですか？",
  "id": "epoch",
  "must_include": [
   "1周"
  ],
  "query": "エポック"
 },
 {
  "group": "seen",
  "question": "「既定モデル」とは何ですか？",
  "id": "default-model",
  "must_include": [
   "gemini-3.5-flash-lite"
  ],
  "query": "既定モデル"
 },
 {
  "group": "seen",
  "question": "「宿題の3レベル」とは何ですか？",
  "id": "homework-levels",
  "must_include": [
   "基礎",
   "どれか1つ"
  ],
  "query": "宿題の3レベル"
 },
 {
  "group": "seen",
  "question": "「Codespaces secret」とは何ですか？",
  "id": "codespaces-secret",
  "must_include": [
   "環境変数"
  ],
  "query": "Codespaces secret"
 },
 {
  "group": "seen",
  "question": "「ハルシネーション」とは何ですか？",
  "id": "hallucination",
  "must_include": [
   "事実と異なる"
  ],
  "query": "ハルシネーション"
 },
 {
  "group": "seen",
  "question": "「TestModel」とは何ですか？",
  "id": "test-model",
  "must_include": [
   "Pydantic AI",
   "テスト"
  ],
  "query": "TestModel"
 },
 {
  "group": "seen",
  "question": "「lecNN-start タグ」とは何ですか？",
  "id": "lec-start-tag",
  "must_include": [
   "タグ",
   "演習"
  ],
  "query": "lecNN-start タグ"
 },
 {
  "group": "seen",
  "question": "「過学習」とは何ですか？",
  "id": "overfitting",
  "must_include": [
   "学習データ",
   "未知"
  ],
  "query": "過学習"
 },
 {
  "group": "para",
  "question": "Low-Rank Adaptation はどんな仕組みで学習しますか？",
  "id": "lora",
  "must_include": [
   "低ランク",
   "固定"
  ],
  "query": "Low-Rank Adaptation"
 },
 {
  "group": "para",
  "question": "overfitting に気づくにはどうすればいいですか？",
  "id": "overfitting",
  "must_include": [
   "検証データ"
  ],
  "query": "overfitting"
 },
 {
  "group": "para",
  "question": "AIE2026 の演習で標準に使うモデルはどれですか？",
  "id": "default-model",
  "must_include": [
   "gemini-3.5-flash-lite"
  ],
  "query": "既定モデル"
 },
 {
  "group": "para",
  "question": "AIE2026 の宿題は、いくつのレベルのうち何個やれば完了ですか？",
  "id": "homework-levels",
  "must_include": [
   "どれか1つ"
  ],
  "query": "宿題"
 },
 {
  "group": "para",
  "question": "幻覚（hallucination）が起きやすいのはどんな質問をしたときですか？",
  "id": "hallucination",
  "must_include": [
   "知らない"
  ],
  "query": "幻覚"
 },
 {
  "group": "para",
  "question": "温度パラメータを低くすると出力はどうなりますか？",
  "id": "temperature",
  "must_include": [
   "安定"
  ],
  "query": "温度パラメータ"
 },
 {
  "group": "para",
  "question": "429エラーが返ってくるのはどんなときですか？",
  "id": "rate-limit",
  "must_include": [
   "上限"
  ],
  "query": "429エラー"
 },
 {
  "group": "para",
  "question": "Function calling で関数を実際に実行するのは誰ですか？",
  "id": "tool",
  "must_include": [
   "プログラム側"
  ],
  "query": "Function calling"
 },
 {
  "group": "para",
  "question": "Pydantic AI で実際の LLM を呼ばずにテストするための偽モデルは何ですか？",
  "id": "test-model",
  "must_include": [
   "TestModel"
  ],
  "query": "TestModel"
 },
 {
  "group": "para",
  "question": "lec03-start は何に使うタグですか？",
  "id": "lec-start-tag",
  "must_include": [
   "ブランチ"
  ],
  "query": "lec03-start"
 },
 {
  "group": "unknown",
  "question": "「RAG」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "RAG"
 },
 {
  "group": "unknown",
  "question": "「プロンプトインジェクション」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "プロンプトインジェクション"
 },
 {
  "group": "unknown",
  "question": "「知識蒸留」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "知識蒸留"
 },
 {
  "group": "unknown",
  "question": "「RLHF」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "RLHF"
 },
 {
  "group": "unknown",
  "question": "「lec04-review タグ」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "lec04-review タグ"
 },
 {
  "group": "unknown",
  "question": "AIE2026 の「修了ボーナス課題」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "修了ボーナス課題"
 },
 {
  "group": "unknown",
  "question": "AIE2026 の「講座ポイント制度」とは何ですか？",
  "id": null,
  "must_include": [],
  "query": "講座ポイント制度"
 }
]""")
train_questions = {r["messages"][0]["content"] for r in rows}
for e in EVAL:
    assert (e["question"] in train_questions) == (e["group"] == "seen"), e["question"]
print({g: sum(e["group"] == g for e in EVAL) for g in ["seen", "para", "unknown"]})


{'seen': 10, 'para': 10, 'unknown': 7}


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
DEVICE = "cuda"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True

# 学習データは system なしの user/assistant なので、評価も system なしで揃える
def generate(model, question):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

DECLINE = ["用語集", "記載", "わかりません", "分かりません", "存在しません", "知りません", "確認"]

def evaluate(model, label):
    out = []
    t0 = time.time()
    for e in EVAL:
        raw = generate(model, e["question"])
        if e["group"] == "unknown":
            ok = any(w in raw for w in DECLINE)   # 用語集にない語には、知らないと言えたら正解
            kind = "declined" if ok else "asserted"
        else:
            ok = all(w in raw for w in e["must_include"])
            kind = ""
        out.append({**e, "raw": raw, "ok": ok, "kind": kind})
    summary = {"label": label, "seconds": round(time.time() - t0, 1)}
    for g in ["seen", "para", "unknown"]:
        rs = [r for r in out if r["group"] == g]
        summary[f"{g}_rate"] = round(sum(r["ok"] for r in rs) / len(rs), 3)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": out}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, out

before_summary, before_rows = evaluate(model, "before")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "seconds": 202.9, "seen_rate": 0.0, "para_rate": 0.0, "unknown_rate": 0.0}


In [ ]:

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
train_ds = Dataset.from_list(rows)   # make_sft_dataset.py の messages 形式をそのまま渡す
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=15, save_strategy="no",
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
train_seconds = round(time.time() - t0)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {train_seconds} 秒 | ステップ数: {trainer.state.global_step} | loss: {losses}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
15,3.129029
30,1.854617
45,0.953258
60,0.394981
75,0.184271
90,0.128055
105,0.101578
120,0.091889
135,0.085048
150,0.082421


学習時間: 172 秒 | ステップ数: 150 | loss: [3.1290293375651044, 1.854616673787435, 0.9532580693562825, 0.3949809392293294, 0.18427057266235353, 0.12805519898732504, 0.10157802899678549, 0.0918888807296753, 0.08504831790924072, 0.08242058753967285]


In [ ]:

after_summary, after_rows = evaluate(tuned, "after")
for b, a in zip(before_rows, after_rows):
    print(f"[{b['group']}] {b['question']}\n  before({'o' if b['ok'] else 'x'}{b['kind']}): {b['raw'][:150]!r}\n  after ({'o' if a['ok'] else 'x'}{a['kind']}): {a['raw'][:150]!r}")
result = {"config": CONFIG, "train_seconds": train_seconds, "losses": losses, "global_step": trainer.state.global_step,
          "before": before_summary, "after": after_summary, "executed_training_myself": True}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))


{"label": "after", "seconds": 95.0, "seen_rate": 1.0, "para_rate": 0.2, "unknown_rate": 0.143}
[seen] 「LoRA」とは何ですか？
  before(x): '**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられま'
  after (o): '事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。'
[seen] 「QLoRA」とは何ですか？
  before(x): '**QLoRA** は、**Qwen3.5** という大規模言語モデル（LLM）の一種です。\n\nQwen3.5 は、Qwen3.5 の進化版であり、**Qwen3.5 の 1.5% のモデルサイズ**です。このモデルは、Qwen3.5 の特徴を維持しつつ、より**軽量**かつ**高速**なパフォーマ'
  after (o): 'ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。'
[seen] 「エポック」とは何ですか？
  before(x): '**「エポック」**（Epoch）は、英語の **Epoch** の日本語訳で、**「時代」「時代間」「時代間期」**を指します。\n\nこの言葉は主に**歴史**や**時間軸**において、特定の時期を指す言葉として使われます。その意味は以下の通りです。\n\n*   **歴史的な意味**: 特定の時代間（'
  after (o): '学習データ全体を1周学習することを1エポックと数える。回数を増やしす